# KNN

Complete KNN fitting and evaluation for use in the final report.

## Setup

In [1]:
from copy import deepcopy
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from mysklearn.mypytable import MyPyTable
from mysklearn.myutils import *
from mysklearn.myclassifiers import *
from mysklearn.myevaluation import *

/home/damon/.local/lib/python3.10/site-packages/matplotlib/projections/__init__.py:63: UserWarning: Unable to import Axes3D. This may be due to multiple versions of Matplotlib being installed (e.g. as a system package and as a pip package). As a result, the 3D projection is not available.
  warnings.warn("Unable to import Axes3D. This may be due to multiple versions of "


In [2]:
CLASSES = ["Pastry","Z_Scratch","K_Scratch","Stains","Dirtiness","Bumps","Other_Faults"]
CLASS_HEADER = "Class"

data = MyPyTable().load_from_file("plate-data.csv")
data = onehot_to_categorical(data, CLASSES)

features = deepcopy(data.column_names)
features.pop(features.index(CLASS_HEADER))

'Class'

## KNN

In [3]:
normalized = MyPyTable(data.column_names, data.data)

for feature in features:
    if feature == "TypeOfSteel_A300" or feature == "TypeOfSteel_A400":
        continue

    col_values = data.get_column(feature)
    normalized.replace_column(feature, normalize_zscore(col_values))

In [4]:
x_data = [[r[0], r[8], r[9], r[15], r[16], r[21], r[25]] for r in normalized.data]
y_data = normalized.get_column(CLASS_HEADER)

folds = stratified_kfold_split(x_data, y_data, n_splits=10, random_state=42, shuffle=True)

In [5]:
for i in range(1, 7):
    print(f"K={i}")
    y_pred, y_actual = [], []

    for fold in folds:
        x_train = [x_data[i] for i in fold[0]]
        x_test = [x_data[i] for i in fold[1]]
        y_train = [y_data[i] for i in fold[0]]
        y_test = [y_data[i] for i in fold[1]]

        knn = MyKNeighborsClassifier(i)
        knn.fit(x_train, y_train)
        y_pred.extend(knn.predict(x_test))
        y_actual.extend(y_test)

    print(accuracy_score(y_actual, y_pred, CLASSES))
    print(recall_score(y_actual, y_pred, CLASSES))
    print(precision_score(y_actual, y_pred, CLASSES))
    print(f1_score(y_actual, y_pred, CLASSES))

K=1
0.8856259659969089
0.6204932152253301
0.6084076657460527
0.6137688948207856
K=2
0.881651578714948
0.5797445294139232
0.6381592206321665
0.5820733326671134
K=3
0.8859203650548318
0.6247521390734906
0.640763329720464
0.627499376656387
K=4
0.8912195480974461
0.6279480489497621
0.6434141990421315
0.6302704255724846
K=5
0.8916611466843306
0.6281829776590948
0.6420162314526461
0.6305051717777165
K=6
0.8921027452712151
0.6318760518193233
0.6454787340866207
0.6325795577389639
